In [ ]:
import logging
from dataclasses import dataclass
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import login as hf_login
from qwen_asr import Qwen3ForcedAligner
from qwen_asr.core.transformers_backend.modeling_qwen3_asr import (
    Qwen3ASRThinkerCausalLMOutputWithPast,
)
from qwen_asr.core.transformers_backend.processing_qwen3_asr import Qwen3ASRProcessor
from transformers import (
    EarlyStoppingCallback,
    EvalPrediction,
    Trainer,
    TrainingArguments,
    set_seed,
)

In [ ]:
MODEL_BASE = "Qwen/Qwen3-ForcedAligner-0.6B"
MODEL_OUTPUT_DIR = "Qwen3-ForcedAligner-0.6B-karaoke-ja-Latn"

MAX_DURATION_SECONDS = 150

TRAIN_EPOCHS = 10
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 2e-4

logging.getLogger("httpx").setLevel(logging.WARNING)
set_seed(42)

In [ ]:
hf_login()

In [ ]:
dataset = load_dataset("NextFire/karaoke-alignments", split="train")

dataset = dataset.map(
    lambda morae: {
        "morae": sorted(
            [mora for mora in morae if mora["value"].strip()],
            key=lambda mora: mora["start"],
        )
    },
    input_columns=["morae"],
    writer_batch_size=500,
    new_fingerprint="qwen3_sort",
)
dataset = dataset.filter(
    lambda morae: (
        len(morae) > 0
        and all(morae[i]["end"] <= morae[i + 1]["start"] for i in range(len(morae) - 1))
    ),
    input_columns=["morae"],
    new_fingerprint="qwen3_overlap",
)
dataset = dataset.filter(
    lambda audio: audio.metadata.duration_seconds <= MAX_DURATION_SECONDS,
    input_columns=["audio"],
    new_fingerprint=f"qwen3_{MAX_DURATION_SECONDS}",
)

In [ ]:
wrapper = Qwen3ForcedAligner.from_pretrained(MODEL_BASE, device_map="auto")
processor = wrapper.processor
model = wrapper.model

In [ ]:
model.thinker._tied_weights_keys = {
    model.thinker._tied_weights_keys[i]: model.thinker._tied_weights_keys[i - 1]  # pyright: ignore[reportArgumentType]
    for i in range(1, len(model.thinker._tied_weights_keys), 2)
}


def _get_input_embeddings():
    return model.thinker.audio_tower.conv2d1


def _set_input_embeddings(value):
    model.thinker.audio_tower.conv2d1 = value


model.thinker.audio_tower.get_input_embeddings = _get_input_embeddings
model.thinker.audio_tower.set_input_embeddings = _set_input_embeddings


def patch_forward(forward):
    def _forward(
        input_ids=None,
        input_features=None,
        attention_mask=None,
        feature_attention_mask=None,
        audio_feature_lengths=None,
        position_ids=None,
        past_key_values=None,
        inputs_embeds=None,
        rope_deltas=None,
        labels=None,
        use_cache=None,
        **kwargs,
    ):
        outputs = forward(
            input_ids=input_ids,
            input_features=input_features,
            attention_mask=attention_mask,
            feature_attention_mask=feature_attention_mask,
            audio_feature_lengths=audio_feature_lengths,
            position_ids=position_ids,
            past_key_values=past_key_values,
            inputs_embeds=inputs_embeds,
            rope_deltas=rope_deltas,
            labels=None,
            use_cache=False,
        )
        assert isinstance(outputs, Qwen3ASRThinkerCausalLMOutputWithPast)
        assert outputs.logits is not None
        assert labels is not None
        num_items_in_batch = kwargs.get("num_items_in_batch")
        loss = F.cross_entropy(
            outputs.logits.float().transpose(1, 2),
            labels,
            ignore_index=-100,
            reduction="sum" if num_items_in_batch is not None else "mean",
        )
        if num_items_in_batch is not None:
            loss /= num_items_in_batch
        return Qwen3ASRThinkerCausalLMOutputWithPast(
            loss=loss,  # pyright: ignore[reportArgumentType]
            logits=outputs.logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
            past_key_values=outputs.past_key_values,
            rope_deltas=outputs.rope_deltas,
        )

    return _forward


model.thinker.forward = patch_forward(model.thinker.forward)

In [ ]:
# for param in model.thinker.audio_tower.parameters():
#     param.requires_grad = False
# for param in model.thinker.model.parameters():
#     param.requires_grad = False
# for param in model.thinker.lm_head.parameters():
#     param.requires_grad = False

In [ ]:
@dataclass
class DataCollatorForForcedAligner:
    processor: Qwen3ASRProcessor
    dynamic_slot_prob: float = 0.5

    def __call__(self, examples: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        all_texts: list[str] = []
        all_audios = []
        all_timestamps: list[list[int]] = []

        for example in examples:
            text, audio, timestamps = self._extract(
                example,
                dynamic_slot_prob=self.dynamic_slot_prob,
            )
            all_texts.append(text)
            all_audios.append(audio)
            all_timestamps.append(timestamps)

        inputs = processor(
            text=all_texts,  # pyright: ignore[reportArgumentType]
            audio=all_audios,
            padding=True,
            return_tensors="pt",
        )

        labels = torch.full_like(inputs["input_ids"], -100)
        for i, (in_row, ts_row) in enumerate(zip(inputs["input_ids"], all_timestamps)):
            ts_pos = (in_row == model.config.timestamp_token_id).nonzero().view(-1)
            n = min(len(ts_pos), len(ts_row))
            labels[i, ts_pos[:n]] = torch.tensor(ts_row[:n], dtype=labels.dtype)

        return {**inputs, "labels": labels}

    def _extract(self, example: dict[str, Any], *, dynamic_slot_prob: float):
        apply_dynamic = np.random.random() < dynamic_slot_prob

        text_parts: list[str] = []
        timestamps: list[int] = []

        for mora in example["morae"]:
            text_part = mora["value"]
            if (not apply_dynamic) or (np.random.random() < 0.5):
                start_idx = int(mora["start"] / model.config.timestamp_segment_time)
                end_idx = int(mora["end"] / model.config.timestamp_segment_time)
                if end_idx <= start_idx:
                    continue
                text_part += "<timestamp><timestamp>"
                timestamps.append(start_idx)
                timestamps.append(end_idx)
            text_parts.append(text_part)

        if apply_dynamic and len(timestamps) == 0:
            return self._extract(example, dynamic_slot_prob=0.0)

        text = "<|audio_start|><|audio_pad|><|audio_end|>" + "".join(text_parts)
        audio = example["audio"]["array"]

        return text, audio, timestamps

In [ ]:
class ForcedAlignerTrainer(Trainer):
    def get_eval_dataloader(self, eval_dataset=None):
        train_data_collator = self.data_collator
        self.data_collator = DataCollatorForForcedAligner(
            self.processing_class,  # pyright: ignore[reportArgumentType]
            dynamic_slot_prob=0.0,
        )
        loader = super().get_eval_dataloader(eval_dataset)
        self.data_collator = train_data_collator
        return loader


def compute_metrics(eval_pred: EvalPrediction):
    preds = eval_pred.predictions
    labels = eval_pred.label_ids
    mask = labels != -100
    aas_frames = np.mean(np.abs(preds[mask] - labels[mask]))
    aas_ms = aas_frames * model.config.timestamp_segment_time
    return {"aas_ms": float(aas_ms)}

In [ ]:
split = dataset.train_test_split(test_size=0.1)

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    remove_unused_columns=False,
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    bf16=True,
    tf32=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    eval_on_start=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=5,
    metric_for_best_model="aas_ms",
    greater_is_better=False,
    load_best_model_at_end=True,
    logging_steps=1,
    report_to="trackio",
)

trainer = ForcedAlignerTrainer(
    model=model.thinker,
    data_collator=DataCollatorForForcedAligner(processor),
    args=training_args,
    preprocess_logits_for_metrics=lambda logits, _: logits[0].argmax(dim=-1),
    compute_metrics=compute_metrics,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

In [ ]:
trainer.train()

In [ ]:
processor.save_pretrained(MODEL_OUTPUT_DIR)

model.thinker.config.__class__.model_type = "qwen3_forced_aligner"
model.save_pretrained(MODEL_OUTPUT_DIR)